# `pyEQL` Tutorial: Charge balancing

![pyEQL Logo](../../pyeql-logo.png)

`pyEQL` is an open-source `python` library for solution chemistry calculations and ion properties developed by the [Kingsbury Lab](https://www.kingsburylab.org/) at Princeton University.

[Documentation](https://pyeql.readthedocs.io/en/latest/) | [How to Install](https://pyeql.readthedocs.io/en/latest/installation.html) | [GitHub](https://github.com/rkingsbury/pyEQL) 

This tutorial introduces strategies for handling charge balance in a `Solution` object.

### Charge balancing strategies
- Approach 0: `None (default)` in which case no charge balancing will be performed
- Approach 1: `auto` which will use the majority cation or anion (i.e., that with the largest concentration) as needed
- Approach 2: `pH` which will adjust the solution pH to balance charge
- Approach 3: `target_ion` which will use a specified cation or anion to balance charge
<!-- - `pE` (not currently implemented) which will adjust the redox equilibrium to balance charge, or the name of a dissolved species e.g. 'Ca+2' or 'Cl-' that will be added/subtracted to balance charge. -->

### Important note between `balance_charge` and `charge_balance`:
- The `balance_charge` *kwarg* controls how charge balancing is performed.
- The `charge_balance` *property* returns the net electric charge of the solution.

### Import `Solution` class

In [1]:
from pyEQL import Solution

In [2]:
from pprint import pprint

### Example of a non-charge balanced solution.

In [3]:
ion_dict = {"Na+": "0.1 mol/L", "K+": "0.2 mol/L", "SO4-2": "0.5 mol/L"}

### **Approach 0:** `balance_charge` = `None (default)`

In [4]:
sol = Solution(solutes=ion_dict, pH=7)
pprint(f"Solution charge balance: {sol.charge_balance}")

'Solution charge balance: -0.7000000000000001'


In [5]:
pprint(sol.components)

{'H2O(aq)': 54.72584323986092,
 'H[+1]': 1e-07,
 'K[+1]': 0.2,
 'Na[+1]': 0.1,
 'OH[-1]': 1.0000000000000001e-07,
 'SO4[-2]': 0.5}


### **Approach 1:** `balance_charge` = "auto"

The `balance_charge = "auto"` will use the majority cation or anion (i.e., that with the largest concentration), in this case `K[+1]`.

In [6]:
sol1 = Solution(solutes=ion_dict, pH=7, balance_charge="auto")
pprint(f"Balance charge: {sol1.balance_charge}, Solution charge balance: {sol1.charge_balance}")

'Balance charge: auto, Solution charge balance: 1.942890160745126e-16'


In [7]:
pprint(sol1.components)

{'H2O(aq)': 54.72584323986092,
 'H[+1]': 1e-07,
 'K[+1]': 0.9000000000000001,
 'Na[+1]': 0.1,
 'OH[-1]': 1.0000000000000001e-07,
 'SO4[-2]': 0.5}


### **Approach 2:** `balance_charge` = "pH"

In [8]:
sol2 = Solution(solutes=ion_dict, pH=7, balance_charge="pH")
pprint(f"Adjusted pH: {sol2.pH}, Solution charge balance: {sol2.charge_balance}")

('Adjusted pH: 0.15490195998573425, Solution charge balance: '
 '1.1942945879741522e-16')


In [9]:
pprint(sol2.components)

{'H2O(aq)': 54.86418501706463,
 'H[+1]': 0.7000000000000144,
 'K[+1]': 0.2,
 'Na[+1]': 0.1,
 'OH[-1]': 1.428571428571399e-14,
 'SO4[-2]': 0.5}


### **Approach 3:** `balance_charge` = "target ion"
Charge balance can also be done by designating a particular "target ion", in this case `Mg[+2]`.

In [10]:
try:
    sol3 = Solution(solutes=ion_dict, pH=7, balance_charge="Mg+2")
    print(f"Solution charge balance: {sol3.charge_balance}")
except Exception as e:
    print(e)

Charge balancing species Mg[+2] was not found in the solution!. Species {'OH[-1]', 'Na[+1]', 'SO4[-2]', 'H[+1]', 'K[+1]'} were found.


Notice: To use `balance_charge`, the target ion must be present in the `Solution` object. 
We add a trace concentration (0.5 mol/L).

In [11]:
# Add target ion in Solution object with 0.5 mol/L
ion_dict["Mg+2"] = "0.5 mol/L"
try:
    sol3 = Solution(solutes=ion_dict, pH=7, balance_charge="Mg+2")
    print(f"Balance charge: {sol3.balance_charge}, Solution charge balance: {sol3.charge_balance}")
except Exception as e:
    print(e)

Balance charge: Mg[+2], Solution charge balance: -1.1102231569740545e-16


In [12]:
pprint(sol3.components)

{'H2O(aq)': 55.231684730030274,
 'H[+1]': 1e-07,
 'K[+1]': 0.2,
 'Mg[+2]': 0.35,
 'Na[+1]': 0.1,
 'OH[-1]': 1.0000000000000001e-07,
 'SO4[-2]': 0.5}


Notice: Equilibration or `equilibrate()` may adjust the solution’s charge balance, regardless of the charge‑balancing method selected.

In [13]:
sol3.equilibrate()
print(f"Balance charge: {sol3.balance_charge}, Solution charge balance: {sol3.charge_balance}")

Balance charge: Mg[+2], Solution charge balance: 1.0949150222941115e-14


In [14]:
pprint(sol3.components)

{'H2(aq)': 6.68016685019436e-35,
 'H2O(aq)': 55.231684730030274,
 'H2SO4(aq)': 2.955381123583309e-17,
 'HSO4[-1]': 4.694880409629213e-07,
 'H[+1]': 1.2355429818950996e-07,
 'KHSO4(aq)': 2.230659291147991e-09,
 'KOH(aq)': 3.4346168841829974e-09,
 'KSO4[-1]': 0.037892500433198605,
 'K[+1]': 0.16210749390154605,
 'MgOH[+1]': 7.556375003093858e-16,
 'MgSO4(aq)': 0.2521686258600411,
 'Mg[+2]': 0.09783115190466592,
 'NaOH(aq)': 9.057306224711908e-10,
 'NaSO4[-1]': 0.017061737901742306,
 'Na[+1]': 0.08293826119252737,
 'O2(aq)': 8.24736736273411e-25,
 'OH[-1]': 1.4646206798488167e-07,
 'SO4[-2]': 0.1928766640863217}
